## Установка библиотек

In [ ]:
# %pip install numpy plotly pandas scikit-learn

You should consider upgrading via the '/Users/vsevolodpanteleev/Projects/dvfu/7-semester/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## Импорты

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

from linreg import *
from decision_tree import *
from random_forest import *

## Генерация синтетических данных

In [ ]:

np.random.seed(42)

n_samples = 500
noise_level = 10.0

X_linear = np.random.uniform(-50, 50, n_samples).reshape(-1, 1)
y_linear = 2.5 * X_linear.flatten() + 10 + np.random.normal(0, noise_level, n_samples)

X_nonlinear = np.random.uniform(-10, 10, n_samples).reshape(-1, 1)
y_nonlinear = 0.5 * X_nonlinear.flatten()**2 + 2 * X_nonlinear.flatten() + 5 + np.random.normal(0, noise_level, n_samples)

print(f"Размер датасета (линейный): {X_linear.shape[0]} образцов")
print(f"Размер датасета (нелинейный): {X_nonlinear.shape[0]} образцов")
print(f"Уровень шума: {noise_level}")

Размер датасета (линейный): 500 образцов
Размер датасета (нелинейный): 500 образцов
Уровень шума: 10.0


## Визуализация исходных данных

In [32]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Линейная зависимость', 'Нелинейная зависимость')
)

fig.add_trace(
    go.Scatter(x=X_linear.flatten(), y=y_linear, mode='markers', name='Линейные данные',
               marker=dict(color='blue', size=5, opacity=0.6)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=X_nonlinear.flatten(), y=y_nonlinear, mode='markers', name='Нелинейные данные',
               marker=dict(color='red', size=5, opacity=0.6)),
    row=1, col=2
)

fig.update_xaxes(title_text="X", row=1, col=1)
fig.update_xaxes(title_text="X", row=1, col=2)
fig.update_yaxes(title_text="Y", row=1, col=1)
fig.update_yaxes(title_text="Y", row=1, col=2)

fig.update_layout(height=400, showlegend=True, title_text="Генерированные данные")
fig.show()

## Тестирование линейной регрессии на линейных данных

In [ ]:

anal_model = CustomAnalLinReg(target=y_linear, features=X_linear)
anal_model.fit(target_error_value=0.1, error=MSE())

grad_model = CustomComputeLinReg(target=y_linear, features=X_linear)
grad_model.fit(target_error_value=0.1, error=MSE(), max_iter=10000, learning_rate=0.0001)

y_pred_anal = anal_model.predict(X_linear)
y_pred_grad = grad_model.predict(X_linear)

mse_anal = MSE().calculate(y_linear, y_pred_anal)
mse_grad = MSE().calculate(y_linear, y_pred_grad)
mae_anal = MAE().calculate(y_linear, y_pred_anal)
mae_grad = MAE().calculate(y_linear, y_pred_grad)

print("=== Линейная регрессия на линейных данных ===")
print(f"Аналитическая модель - MSE: {mse_anal:.4f}, MAE: {mae_anal:.4f}")
print(f"Градиентная модель - MSE: {mse_grad:.4f}, MAE: {mae_grad:.4f}")
print(f"\nВеса (аналитическая): {anal_model.weights}")
print(f"Смещение (аналитическая): {anal_model.bias:.4f}")
print(f"Веса (градиентная): {grad_model.weights}")
print(f"Смещение (градиентная): {grad_model.bias:.4f}")

=== Линейная регрессия на линейных данных ===
Аналитическая модель - MSE: 100.7459, MAE: 8.0592
Градиентная модель - MSE: 102.6039, MAE: 8.1066

Веса (аналитическая): [2.52137226]
Смещение (аналитическая): 10.0793
Веса (градиентная): [2.52277989]
Смещение (градиентная): 8.7155


## Визуализация результатов линейной регрессии

In [ ]:

sort_idx = np.argsort(X_linear.flatten())
X_sorted = X_linear[sort_idx]
y_pred_anal_sorted = y_pred_anal[sort_idx]
y_pred_grad_sorted = y_pred_grad[sort_idx]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X_linear.flatten(), y=y_linear,
    mode='markers', name='Исходные данные',
    marker=dict(color='lightblue', size=5, opacity=0.5)
))

fig.add_trace(go.Scatter(
    x=X_sorted.flatten(), y=y_pred_anal_sorted,
    mode='lines', name='Аналитическая модель',
    line=dict(color='green', width=3)
))

fig.add_trace(go.Scatter(
    x=X_sorted.flatten(), y=y_pred_grad_sorted,
    mode='lines', name='Градиентная модель',
    line=dict(color='orange', width=3, dash='dash')
))

fig.update_layout(
    title='Линейная регрессия на линейных данных',
    xaxis_title='X',
    yaxis_title='Y',
    height=500
)

fig.show()

## Тестирование моделей на нелинейных данных

In [ ]:

linreg_nl = CustomAnalLinReg(target=y_nonlinear, features=X_nonlinear)
linreg_nl.fit(target_error_value=0.1, error=MSE())
y_pred_linreg_nl = linreg_nl.predict(X_nonlinear)

tree_model = DecisionTreeRegressor(max_depth=10, min_samples_split=5, min_samples_leaf=2)
tree_model.fit(X_nonlinear, y_nonlinear)
y_pred_tree = tree_model.predict(X_nonlinear)

forest_model = RandomForestRegressor(
    n_estimators=50, 
    max_depth=10, 
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42
)
forest_model.fit(X_nonlinear, y_nonlinear)
y_pred_forest = forest_model.predict(X_nonlinear)

mse_linreg = MSE().calculate(y_nonlinear, y_pred_linreg_nl)
mse_tree = MSE().calculate(y_nonlinear, y_pred_tree)
mse_forest = MSE().calculate(y_nonlinear, y_pred_forest)

mae_linreg = MAE().calculate(y_nonlinear, y_pred_linreg_nl)
mae_tree = MAE().calculate(y_nonlinear, y_pred_tree)
mae_forest = MAE().calculate(y_nonlinear, y_pred_forest)

print("=== Сравнение моделей на нелинейных данных ===")
print(f"Линейная регрессия - MSE: {mse_linreg:.4f}, MAE: {mae_linreg:.4f}")
print(f"Дерево решений     - MSE: {mse_tree:.4f}, MAE: {mae_tree:.4f}")
print(f"Случайный лес      - MSE: {mse_forest:.4f}, MAE: {mae_forest:.4f}")

=== Сравнение моделей на нелинейных данных ===
Линейная регрессия - MSE: 341.6534, MAE: 15.1679
Дерево решений     - MSE: 50.7815, MAE: 5.6708
Случайный лес      - MSE: 45.7550, MAE: 5.4538


## Визуализация сравнения моделей

In [ ]:

sort_idx_nl = np.argsort(X_nonlinear.flatten())
X_nl_sorted = X_nonlinear[sort_idx_nl]
y_pred_linreg_sorted = y_pred_linreg_nl[sort_idx_nl]
y_pred_tree_sorted = y_pred_tree[sort_idx_nl]
y_pred_forest_sorted = y_pred_forest[sort_idx_nl]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X_nonlinear.flatten(), y=y_nonlinear,
    mode='markers', name='Исходные данные',
    marker=dict(color='lightcoral', size=5, opacity=0.4)
))

fig.add_trace(go.Scatter(
    x=X_nl_sorted.flatten(), y=y_pred_linreg_sorted,
    mode='lines', name=f'Линейная регрессия (MSE: {mse_linreg:.2f})',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=X_nl_sorted.flatten(), y=y_pred_tree_sorted,
    mode='lines', name=f'Дерево решений (MSE: {mse_tree:.2f})',
    line=dict(color='green', width=2)
))

fig.add_trace(go.Scatter(
    x=X_nl_sorted.flatten(), y=y_pred_forest_sorted,
    mode='lines', name=f'Случайный лес (MSE: {mse_forest:.2f})',
    line=dict(color='purple', width=2, dash='dash')
))

fig.update_layout(
    title='Сравнение моделей на нелинейных данных',
    xaxis_title='X',
    yaxis_title='Y',
    height=600
)

fig.show()

## Сравнение метрик

In [ ]:

metrics_df = pd.DataFrame({
    'Модель': ['Линейная регрессия', 'Дерево решений', 'Случайный лес'],
    'MSE': [mse_linreg, mse_tree, mse_forest],
    'MAE': [mae_linreg, mae_tree, mae_forest]
})

print("\n=== Сводная таблица метрик ===")
print(metrics_df.to_string(index=False))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('MSE (меньше - лучше)', 'MAE (меньше - лучше)')
)

fig.add_trace(
    go.Bar(x=metrics_df['Модель'], y=metrics_df['MSE'], name='MSE',
           marker=dict(color=['blue', 'green', 'purple'])),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=metrics_df['Модель'], y=metrics_df['MAE'], name='MAE',
           marker=dict(color=['blue', 'green', 'purple'])),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False, title_text="Сравнение метрик моделей")
fig.show()


=== Сводная таблица метрик ===
            Модель        MSE       MAE
Линейная регрессия 341.653412 15.167946
    Дерево решений  50.781483  5.670762
     Случайный лес  45.755016  5.453838


## Генерация данных для классификации

In [ ]:

np.random.seed(42)

n_samples_class = 300

X_class_0 = np.random.randn(n_samples_class, 2) * 1.5 + np.array([-2, -2])
y_class_0 = np.zeros(n_samples_class, dtype=int)

X_class_1 = np.random.randn(n_samples_class, 2) * 1.5 + np.array([2, 2])
y_class_1 = np.ones(n_samples_class, dtype=int)

# Объединяем данные
X_classification = np.vstack([X_class_0, X_class_1])
y_classification = np.hstack([y_class_0, y_class_1])

# Перемешиваем
shuffle_indices = np.random.permutation(len(y_classification))
X_classification = X_classification[shuffle_indices]
y_classification = y_classification[shuffle_indices]

print(f"Размер датасета классификации: {X_classification.shape[0]} образцов")
print(f"Количество признаков: {X_classification.shape[1]}")
print(f"Классы: {np.unique(y_classification)}")
print(f"Распределение классов: {np.bincount(y_classification)}")

Размер датасета классификации: 600 образцов
Количество признаков: 2
Классы: [0 1]
Распределение классов: [300 300]


## Визуализация данных классификации

In [ ]:
fig = go.Figure()

mask_0 = y_classification == 0
fig.add_trace(go.Scatter(
    x=X_classification[mask_0, 0],
    y=X_classification[mask_0, 1],
    mode='markers',
    name='Класс 0',
    marker=dict(color='blue', size=8, opacity=0.6)
))

mask_1 = y_classification == 1
fig.add_trace(go.Scatter(
    x=X_classification[mask_1, 0],
    y=X_classification[mask_1, 1],
    mode='markers',
    name='Класс 1',
    marker=dict(color='red', size=8, opacity=0.6)
))

fig.update_layout(
    title='Генерированные данные для бинарной классификации',
    xaxis_title='Признак 1',
    yaxis_title='Признак 2',
    height=500
)

fig.show()

## Обучение моделей классификации

In [ ]:
tree_clf_gini = DecisionTreeClassifier(max_depth=5, min_samples_split=10, split_metric=GiniMetric())
tree_clf_gini.fit(X_classification, y_classification)
y_pred_tree_gini = tree_clf_gini.predict(X_classification)

tree_clf_entropy = DecisionTreeClassifier(max_depth=5, min_samples_split=10, split_metric=EntropyMetric())
tree_clf_entropy.fit(X_classification, y_classification)
y_pred_tree_entropy = tree_clf_entropy.predict(X_classification)

forest_clf = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    min_samples_split=10,
    max_features='sqrt',
    random_state=42
)
forest_clf.fit(X_classification, y_classification)
y_pred_forest_clf = forest_clf.predict(X_classification)

accuracy_tree_gini = np.mean(y_pred_tree_gini == y_classification)
accuracy_tree_entropy = np.mean(y_pred_tree_entropy == y_classification)
accuracy_forest = np.mean(y_pred_forest_clf == y_classification)

print("=== Точность моделей классификации ===")
print(f"Дерево решений (Gini):    {accuracy_tree_gini:.4f}")
print(f"Дерево решений (Entropy): {accuracy_tree_entropy:.4f}")
print(f"Случайный лес:            {accuracy_forest:.4f}")

=== Точность моделей классификации ===
Дерево решений (Gini):    0.9883
Дерево решений (Entropy): 0.9933
Случайный лес:            0.9900


## Визуализация границ решений

In [ ]:
x_min, x_max = X_classification[:, 0].min() - 1, X_classification[:, 0].max() + 1
y_min, y_max = X_classification[:, 1].min() - 1, X_classification[:, 1].max() + 1
h = 0.1 

xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid_points = np.c_[xx.ravel(), yy.ravel()]

Z_forest = forest_clf.predict(grid_points).reshape(xx.shape)

fig = go.Figure()

fig.add_trace(go.Contour(
    x=np.arange(x_min, x_max, h),
    y=np.arange(y_min, y_max, h),
    z=Z_forest,
    colorscale=[[0, 'lightblue'], [1, 'lightcoral']],
    showscale=False,
    opacity=0.3,
    hoverinfo='skip'
))

mask_0 = y_classification == 0
fig.add_trace(go.Scatter(
    x=X_classification[mask_0, 0],
    y=X_classification[mask_0, 1],
    mode='markers',
    name='Класс 0 (истинный)',
    marker=dict(color='blue', size=8, symbol='circle', line=dict(width=1, color='darkblue'))
))

mask_1 = y_classification == 1
fig.add_trace(go.Scatter(
    x=X_classification[mask_1, 0],
    y=X_classification[mask_1, 1],
    mode='markers',
    name='Класс 1 (истинный)',
    marker=dict(color='red', size=8, symbol='circle', line=dict(width=1, color='darkred'))
))

fig.update_layout(
    title=f'Границы решений случайного леса (Accuracy: {accuracy_forest:.2%})',
    xaxis_title='Признак 1',
    yaxis_title='Признак 2',
    height=600,
    width=700
)

fig.show()

## Сравнение моделей классификации

In [42]:

clf_metrics_df = pd.DataFrame({
    'Модель': ['Дерево (Gini)', 'Дерево (Entropy)', 'Случайный лес'],
    'Точность': [accuracy_tree_gini, accuracy_tree_entropy, accuracy_forest]
})

print("\n=== Сводная таблица точности моделей ===")
print(clf_metrics_df.to_string(index=False))

fig = go.Figure()

fig.add_trace(go.Bar(
    x=clf_metrics_df['Модель'],
    y=clf_metrics_df['Точность'],
    text=clf_metrics_df['Точность'].apply(lambda x: f'{x:.2%}'),
    textposition='auto',
    marker=dict(color=['lightblue', 'lightgreen', 'purple'])
))

fig.update_layout(
    title='Сравнение точности моделей классификации',
    xaxis_title='Модель',
    yaxis_title='Точность',
    yaxis=dict(range=[0, 1.1]),
    height=400
)

fig.show()


=== Сводная таблица точности моделей ===
          Модель  Точность
   Дерево (Gini)  0.988333
Дерево (Entropy)  0.993333
   Случайный лес  0.990000
